## Images
Images are nothing more than arrays, black and white images are 2D arrays containing x and y values of brightness. Color images are 3D images with RGB values ie 3 values each representing the amount of red, green and blue in the image.

Let's start by creating an image by ourselves!

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

image = [[0, 0, 0, 0, 0, 0, 0],
         [0, 0, 1, 0, 1, 0, 0],
         [0, 0, 1, 0, 1, 0, 0],
         [0, 0, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 1, 0],
         [0, 0, 1, 1, 1, 0, 0],
         [0, 0, 0, 0, 0, 0, 0]]

plt.imshow(image, cmap='gray', interpolation='nearest')
plt.axis('off') 
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Replace with your image file path
image_path = "res/president.jpg"

# Load the image
img = mpimg.imread(image_path)

# Plot the image
plt.imshow(img)
plt.axis('off')  # Hide axes for cleaner display
plt.show()




And let's try some kernel operations that can be found in AI or in image processing

A single neuron in CNN (convolutional neural network) is a kernel. A matrix with numbers. There are many types of kernels, down here, you can try to detect either horizontal or vertical edge using the following:

here this is a horizontal edge detector, it will detect horizontal edges:
```
kernel = np.array([[-1, -2, -1],
                    [0, 0, 0, ],
                    [1, 2, 1]])
```

here is a vertical edge detector, it will detect vertical edges:
```
kernel = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])
```

In [ ]:
import numpy as np
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

""" kernel = np.array([[-1, 0, 1], # vertical edges
                    [-2, 0, 2],
                    [-1, 0, 1]]) 
"""
kernel = np.array([[-2, -1, -1], # horizontal edges
                    [0, 0, 0],
                    [1, 2, 1]]) 


def apply_kernel_on_segment(kernel, segment):
    return np.sum(kernel * segment)

def apply_kernel_on_image(kernel, image):
    # image: H x W x C (RGB)
    # convert to grayscale
    gray = image.mean(axis=2)

    kh, kw = kernel.shape
    H, W = gray.shape
    out_h = H - kh + 1
    out_w = W - kw + 1

    output = np.zeros((out_h, out_w), dtype=float)

    for i in range(out_h):
        for j in range(out_w):
            patch = gray[i:i+kh, j:j+kw]
            output[i, j] = apply_kernel_on_segment(kernel, patch)

    return output

image = mpimg.imread(image_path).astype(float)
if image.max() > 1.0:
    image /= 255.0

feature_map = apply_kernel_on_image(kernel, image)

plt.imshow(feature_map, cmap='gray')
plt.colorbar()
plt.axis('off')
plt.show()


You can have a kernel that will detect edges in all directions

```
kernel = np.array([[-1, -1, -1],
                    [-1, 8, -1],
                    [-1, -1, -1]])
```

In [ ]:
import numpy as np
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

kernel = np.array([[-1, -1, -1],
                    [-1, 5, -1],
                    [-1, -1, -1]])

def apply_kernel_on_segment(kernel, segment):
    return np.sum(kernel * segment)

def apply_kernel_on_image(kernel, image):
    # image: H x W x C (RGB)
    # convert to grayscale
    gray = image.mean(axis=2)

    kh, kw = kernel.shape
    H, W = gray.shape
    out_h = H - kh + 1
    out_w = W - kw + 1

    output = np.zeros((out_h, out_w), dtype=float)

    for i in range(out_h):
        for j in range(out_w):
            patch = gray[i:i+kh, j:j+kw]
            output[i, j] = apply_kernel_on_segment(kernel, patch)

    return output

image = mpimg.imread(image_path).astype(float)
if image.max() > 1.0:
    image /= 255.0

feature_map = apply_kernel_on_image(kernel, image)

plt.imshow(feature_map, cmap='gray')
plt.colorbar()
plt.axis('off')
plt.show()


# Convolutional Layer

In [ ]:
import numpy as np

def pad2d(x, pad):
    if pad == 0:
        return x
    return np.pad(x, ((0,0),(0,0),(pad,pad),(pad,pad)), mode="constant")

class Conv2D:
    """
    NCHW input.
    W shape: (C_out, C_in, KH, KW)
    """
    def __init__(self, c_in, c_out, k=3, stride=1, padding=1, lr=0.01, seed=0):
        rng = np.random.default_rng(seed)
        self.stride = stride
        self.padding = padding
        self.lr = lr

        self.W = (rng.standard_normal((c_out, c_in, k, k)) * 0.02).astype(np.float32)
        self.b = np.zeros((c_out,), dtype=np.float32)

        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

        self.x = None
        self.xp = None  # padded input

    def forward(self, x):
        self.x = x
        xp = pad2d(x, self.padding)
        self.xp = xp

        N, Cin, H, W = xp.shape
        Cout, _, KH, KW = self.W.shape
        S = self.stride

        Hout = (H - KH) // S + 1
        Wout = (W - KW) // S + 1
        out = np.zeros((N, Cout, Hout, Wout), dtype=np.float32)

        for n in range(N):
            for co in range(Cout):
                for i in range(Hout):
                    hs = i * S
                    for j in range(Wout):
                        ws = j * S
                        patch = xp[n, :, hs:hs+KH, ws:ws+KW]
                        out[n, co, i, j] = np.sum(patch * self.W[co]) + self.b[co]
        return out
 def backward(self, dout):
        xp = self.xp
        N, Cin, H, W = xp.shape
        Cout, _, KH, KW = self.W.shape
        S = self.stride

        _, _, Hout, Wout = dout.shape

        dxp = np.zeros_like(xp, dtype=np.float32)
        self.dW.fill(0.0)
        self.db.fill(0.0)

        for n in range(N):
            for co in range(Cout):
                self.db[co] += np.sum(dout[n, co])
                for i in range(Hout):
                    hs = i * S
                    for j in range(Wout):
                        ws = j * S
                        g = dout[n, co, i, j]
                        patch = xp[n, :, hs:hs+KH, ws:ws+KW]
                        self.dW[co] += g * patch
                        dxp[n, :, hs:hs+KH, ws:ws+KW] += g * self.W[co]
        # unpad
        if self.padding > 0:
            dx = dxp[:, :, self.padding:-self.padding, self.padding:-self.padding]
        else:
            dx = dxp
        return dx

    def step(self):
        self.W -= self.lr * self.dW
        self.b -= self.lr * self.db



# Activation Layer
'max pool' instead of ReLu => find highest number in a convolution (pool)
- **Stride N**: instead of moving by one we move by groups of n => reduced matrix size
$$
X=\begin{bmatrix}
1 & 3 & 2 & 1\\
4 &6  &5& 2\\
1 &2& 0 &1\\
3 &1  &2&4 \\
\end{bmatrix}
=>\
\begin{bmatrix}
3 & 2\\
6 &5\\
2&1\\
3 &4 \\
\end{bmatrix}
=>\
\begin{bmatrix}
6 & 5\\
3 & 4
\end{bmatrix}


In [ ]:


class MaxPool2x2:
    def __init__(self):
        self.x = None
        self.argmax = None

    def forward(self, x):
        # x: (N, C, H, W), assume H and W even
        self.x = x
        N, C, H, W = x.shape
        out = np.zeros((N, C, H//2, W//2), dtype=np.float32)
        self.argmax = np.zeros_like(out, dtype=np.int32)

        for n in range(N):
            for c in range(C):
                for i in range(H//2):
                    hs = i * 2
                    for j in range(W//2):
                        ws = j * 2
                        patch = x[n, c, hs:hs+2, ws:ws+2].reshape(-1)
                        idx = int(np.argmax(patch))
                        out[n, c, i, j] = patch[idx]
                        self.argmax[n, c, i, j] = idx
        return out

    def backward(self, dout):
        x = self.x
        N, C, H, W = x.shape
        dx = np.zeros_like(x, dtype=np.float32)

        for n in range(N):
            for c in range(C):
                for i in range(H//2):
                    hs = i * 2
                    for j in range(W//2):
                        ws = j * 2
                        idx = self.argmax[n, c, i, j]
                        # map idx 0..3 back into 2x2
                        di, dj = divmod(idx, 2)
                        dx[n, c, hs+di, ws+dj] += dout[n, c, i, j]
        return dx

class SoftmaxCrossEntropy:
    def __init__(self):
        self.probs = None
        self.y = None

    def forward(self, logits, y):
        # logits: (N, K), y: (N,) int labels
        logits = logits - np.max(logits, axis=1, keepdims=True)
        exp = np.exp(logits)
        probs = exp / np.sum(exp, axis=1, keepdims=True)

        N = logits.shape[0]
        loss = -np.mean(np.log(probs[np.arange(N), y] + 1e-12))

        self.probs = probs
        self.y = y
        return loss, probs

    def backward(self):
        # dL/dlogits
        probs = self.probs.copy()
        N = probs.shape[0]
        probs[np.arange(N), self.y] -= 1.0
        return probs / N

